# CS521 HW3 - Problem 2

Jinwu Hu (jinwuhu2)

This notebook contains the complete implementation. Select a GPU runtime and run the cells in order. Only public Python packages and Microsoft's Phi-2 model are downloaded; no external source files are needed.

In [ ]:
%pip install -q "syncode==0.4.16" "transformers==4.53.2" "accelerate==1.15.0"


## Method

Four few-shot examples show only extraction. SynCode constrains Phi-2 to a numeric record with exactly two ordered operands and one operator. The program safely parses the record using `ast.literal_eval` and uses Python arithmetic to compute the answer. The number grammar allows a trailing decimal point so incomplete decimal prefixes remain live; see [SynCode issue #195](https://github.com/structuredllm/syncode/issues/195). This record syntax is not restricted to strict JSON. The required entry point is `run_syncode(textual_query)`; it returns `(list, str, float)`.

References: [SynCode](https://github.com/structuredllm/syncode), [Phi-2](https://huggingface.co/microsoft/phi-2).

In [ ]:
"""CS521 HW3 Q2: Phi-2 extraction with SynCode and external Python arithmetic.

This module is embedded in full in the self-contained notebook.
"""
import json
import ast
import os
import operator
import time

MODEL_REVISION = "810d367871c1d460086d9f82db8696f2e0a0fcd0"

GRAMMAR = r'''
start: "{" "\"operands\"" ":" "[" NUMBER "," NUMBER "]" "," "\"operator\"" ":" OP "}"
NUMBER: /-?(0|[1-9][0-9]*)(\.[0-9]*)?/
OP: "\"+\"" | "\"-\"" | "\"*\"" | "\"/\""
%import common.WS
%ignore WS
'''

# These examples contain extraction only, never the computed result.
FEW_SHOT = '''Extract the two operands in calculation order and the one binary operator.
Return only a record with operands and operator. Use +, -, *, or /.
Represent each operand as a decimal number. For "subtract A from B", the
operands must be [B, A]. Do not perform the calculation.

Query: What is 12.5 plus 7.25?
Extraction: {"operands":[12.5,7.25],"operator":"+"}

Query: Subtract 8.5 from 20.0.
Extraction: {"operands":[20.0,8.5],"operator":"-"}

Query: What is -3.0 multiplied by 6.5?
Extraction: {"operands":[-3.0,6.5],"operator":"*"}

Query: What is 81.0 divided by 9.0?
Extraction: {"operands":[81.0,9.0],"operator":"/"}

Query: '''

_STATE = None
LAST_EXTRACTION = None
OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}

def _initialize():
    global _STATE
    if _STATE is not None:
        return _STATE
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from syncode import SyncodeLogitsProcessor
    from syncode.parsers.grammars import Grammar
    torch.set_num_threads(int(os.environ.get("CS521_CPU_THREADS", "4")))
    device = os.environ.get("CS521_DEVICE") or (
        "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    path = os.environ.get("CS521_PHI2_PATH", "microsoft/phi-2")
    tokenizer = AutoTokenizer.from_pretrained(
        path, revision=MODEL_REVISION, trust_remote_code=False)
    # RTX 4090 supports BF16. On a Colab T4, use FP32 to avoid unsupported
    # BF16 kernels and Phi-2's known FP16 attention-overflow sensitivity.
    dtype = (torch.bfloat16
             if device.startswith("cuda") and torch.cuda.get_device_capability(device)[0] >= 8
             else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(
        path, revision=MODEL_REVISION, trust_remote_code=False,
        torch_dtype=dtype,
        attn_implementation="eager",
    ).to(device).eval()
    processor = SyncodeLogitsProcessor(
        Grammar(GRAMMAR), tokenizer, parse_output_only=True,
        mode="grammar_strict", dev_mode=True,
    )
    _STATE = (model, tokenizer, processor, device)
    return _STATE

def run_syncode(textual_query):
    """Return (ordered list of operands, operator string, final answer float)."""
    global LAST_EXTRACTION
    if not isinstance(textual_query, str) or not textual_query.strip():
        raise ValueError("textual_query must be a nonempty string")
    import torch
    from transformers import LogitsProcessor
    model, tokenizer, processor, device = _initialize()

    class FiniteLogits(LogitsProcessor):
        def __call__(self, input_ids, scores):
            if not torch.isfinite(scores).all().item():
                raise FloatingPointError("Non-finite Phi-2 logits")
            return scores

    prompt = FEW_SHOT + textual_query.strip() + "\nExtraction:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    processor.reset()
    before = time.perf_counter()
    with torch.inference_mode():
        tokens = model.generate(
            **inputs, max_new_tokens=96, do_sample=False,
            logits_processor=[FiniteLogits(), processor],
            pad_token_id=tokenizer.eos_token_id,
            temperature=None, top_p=None, top_k=None,
        )
    output_ids = tokens[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    # SynCode's incremental lexer can prematurely finish NUMBER when a decimal
    # point temporarily makes the prefix non-accepting (upstream issue #195).
    # Accepting a trailing dot preserves the live numeric prefix. The completed
    # record is a Python numeric literal, not necessarily strict JSON (327. is
    # legal here). literal_eval parses literals only; it cannot execute code.
    extracted = ast.literal_eval(text)
    operands = [float(v) for v in extracted["operands"]]
    op = extracted["operator"]
    if len(operands) != 2 or op not in OPS:
        raise ValueError("Unexpected extraction schema")
    LAST_EXTRACTION = {"query": textual_query, "raw_extraction": text,
                       "generated_tokens": len(output_ids),
                       "seconds": time.perf_counter() - before}
    # Arithmetic is performed only here, by Python, after constrained extraction.
    answer = float(OPS[op](*operands))
    LAST_EXTRACTION["result"] = [operands, op, answer]
    return operands, op, answer

TEST_CASES = [
    ("What is 327. multiplied by 11.0?", [327.0,11.0], "*", 3597.0),
    ("What is 45.1 plus 23.54?", [45.1,23.54], "+", 68.64),
    ("What is 120.4 divided by 4.0?", [120.4,4.0], "/", 30.1),
    ("What is 19.75 minus 6.25?", [19.75,6.25], "-", 13.5),
    ("Subtract 3.2 from 9.7.", [9.7,3.2], "-", 6.5),
    ("What is -2.5 times 4.0?", [-2.5,4.0], "*", -10.0),
    ("Divide -12.6 by 3.0.", [-12.6,3.0], "/", -4.2),
    ("Add 0.125 and 0.375.", [0.125,0.375], "+", 0.5),
    ("What is the difference of 7.5 and 2.25?", [7.5,2.25], "-", 5.25),
    ("What is 1000.0 multiplied by 0.01?", [1000.0,0.01], "*", 10.0),
    ("What is 9 divided by 2?", [9.0,2.0], "/", 4.5),
    ("What is -4.5 plus -3.25?", [-4.5,-3.25], "+", -7.75),
    ("What is 8.5 minus -2.5?", [8.5,-2.5], "-", 11.0),
    ("What is 0.0 multiplied by 725.25?", [0.0,725.25], "*", 0.0),
    ("What is the quotient of 1.0 and 8.0?", [1.0,8.0], "/", 0.125),
    ("What is 65.4321 plus 0.5679?", [65.4321,0.5679], "+", 66.0),
    ("What is 10. minus 1.?", [10.0,1.0], "-", 9.0),
    ("What is the product of 7.25 and 2.0?", [7.25,2.0], "*", 14.5),
    ("What is 0.5 divided by -0.25?", [0.5,-0.25], "/", -2.0),
    ("What is 999.5 plus 0.5?", [999.5,0.5], "+", 1000.0),
]



In [ ]:
for query, operands, op, expected in TEST_CASES[:3]:
    actual = run_syncode(query)
    print(query, "->", actual)


## Recorded validation

The following output summarizes a separate CUDA run of this exact implementation. It is saved evidence, not a claim that this notebook has already run in a new Colab session.

In [ ]:
import math
results = []
for query, operands, op, expected in TEST_CASES:
    actual = run_syncode(query)
    results.append(actual[0] == operands and actual[1] == op and math.isclose(actual[2], expected, abs_tol=1e-9))
print({"tested_queries": len(results), "passed": sum(results)})


In [ ]:
recorded_validation = {'tested_queries': 20, 'passed': 20, 'outputs': [[[327.0, 11.0], '*', 3597.0], [[45.1, 23.54], '+', 68.64], [[120.4, 4.0], '/', 30.1]]}
print(json.dumps(recorded_validation,indent=2))


{
  "tested_queries": 20,
  "passed": 20,
  "outputs": [
    [
      [
        327.0,
        11.0
      ],
      "*",
      3597.0
    ],
    [
      [
        45.1,
        23.54
      ],
      "+",
      68.64
    ],
    [
      [
        120.4,
        4.0
      ],
      "/",
      30.1
    ]
  ]
}
